
![logo](https://raw.githubusercontent.com/pdehidalgo/dbx-ucm-productivizacion/main/logo_ucm.jpg)




Ejecución: 

Crea un Cómputo con **ML** activo y **Runtime** 16.4 LTS y asócialo para ejecutar el Notebook.  



## Contexto de la tarea
En entornos IoT (_internet of things_, por ejemplo, sensórica industrial), cada dispositivo de medición tomará datos específicos de su maquinaria asociada por lo que entrenar un modelo particular por _item_ permite mejorar la precisión global de las predicciones. Este enfoque también puede extenderse a clientes, regiones o productos y también a diferentes dominios como el _retail_ o el marketing. 

Para hacer esto posible y de manera escalable vamos a utilizar la API de Pandas en PySpark (`applyInPandas`) dedicada al entrenamiento de modelos de machine learning. 

## Descripción técnica: entrenamiento usando la API de Funciones de Pandas. En esta tarea se pedirá:<br>
 - Utilización <a href="https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.GroupedData.applyInPandas.html" target="_blank">**.groupBy().applyInPandas()**</a> para construir múltiples modelos en paralelo.
 - Registro modelos con MLflow y asociarlos a cada dispositivo.
 - Agrupar múltiples modelos en un único endpoint de inferencia usando `PythonModel` personalizado.


## Objetivo: 

- Familiarizarse con el uso de MLflow integrado en Databricks.
- Utilización de un modelo base de MLflow y ampliación del mismo.
- Conocimiento sobre la API de Pandas en PySpark, útil en circustancias como las que se piden en la tarea con multitud de modelos (orquestación del entrenamiento) pero datos limitados.


## Comenzamos con la creación del dataset: 

Vamos a generar un conjunto de datos sintéticos aislando así la etapa de lectura de los datos. El objetivo de la práctica no es tanto la generación del _mejor modelo_, que ya habéis trabajado en otras asignaturas sino cómo entrenarlos y donde desplegarlos. El dataset será sintético emulando los dispositivos IoT comentados anteriormente: 

- **`device_id`**: representa 10 dispositivos distintos.
- **`record_id`**: 10.000 registros únicos simulados.
- **`feature_1`**, **`feature_2`**, **`feature_3`**: variables independientes para el entrenamiento del modelo.
- **`label`**: variable objetivo que queremos predecir.


In [0]:
# Entrenamiento con Pandas Function API

# Sección 1: Generación de datos sintéticos para simular entradas por dispositivo IoT
# Usamos mod 10 para simular 10 dispositivos diferentes.
# Mostramos solo las primeras filas.

import pyspark.sql.functions as f

df = (spark
      .range(1000*100)
      .select(f.col("id").alias("record_id"), (f.col("id")%10).alias("device_id"))
      .withColumn("feature_1", f.rand() * 1)
      .withColumn("feature_2", f.rand() * 2)
      .withColumn("feature_3", f.rand() * 3)
      .withColumn("label", (f.col("feature_1") + f.col("feature_2") + f.col("feature_3")) + f.rand())
     )

display(df.limit(5))

record_id,device_id,feature_1,feature_2,feature_3,label
0,0,0.0904273795339996,0.8810617766487288,2.7301811159101637,4.267273758929227
1,1,0.970576559087917,1.6057389005540723,2.0166728360277917,5.014736203307659
2,2,0.8981697540204603,0.3208302434715533,2.7063200019211786,4.179334739851246
3,3,0.7597185243452869,1.998050816871638,2.082651154843672,5.81447546487679
4,4,0.9876138648146251,0.9756198824066087,2.839370708951786,4.932814001244762


In [0]:
# TODO (apartado 1): cuenta el número de registros del df por deviceId

In [0]:
# Sección 2: Esquema de retorno que definirá el formato del resultado tras entrenar cada modelo. El schema es muy importante y un motivo de fallo común cuando estamos trabajando con PandasUDF y Spark. Pensad en el esquema como la interfaz entre ambos sistemas, para que no exista error, los tipos esperados deben coincidir entre el cliente Python y el Dataframe de Spark.

train_return_schema = "device_id integer, n_used integer, model_path string, mse float"

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, FloatType
# TODO (apartado 2): Si queremos realizar la anterior operación con código (best practice) en lugar de un String para el schema, ¿Qué otra opción tenemos para definirlo?

train_return_schema = StructType([
     StructField("device_id", ...


Definamos ahora una función de pandas que reciba todos los datos correspondientes de cada dispositivo, entrene un modelo, lo guarde como una ejecución anidada (_nested run_) para tener cada entrenamiento bajo un mismo experimento y devuelva el resultado. 

**NOTA IMPORTANTE:** en la siguiente función no filtramos por sensor o dispositivo, vamos a crear una función genérica que trabaje a nivel de un único dispositivo y será desde Spark desde donde manejaremos la distribución de los datos. El id de experimento se usará para la ejecución anidada.  


In [0]:
# Sección 3: 

# Loggeo de parámetros y uso de nested runs para trazabilidad. A completar: 
# Validación de esquema, separación train/test y filtrado por MSE.

import mlflow
import mlflow.sklearn
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

def train_model(df_pandas: pd.DataFrame, model_type: str) -> pd.DataFrame:

    columnas_esperadas = {"device_id", "run_id", "feature_1", "feature_2", "feature_3", "label"}

    # TODO (apartado 3): Validación del esquema de entrada: comprueba que todas las columnas esperadas están presentes, de lo contrario, lanza una excepción de tipo ValueError
    ...

    device_id = df_pandas["device_id"].iloc[0]
    n_used = df_pandas.shape[0]
    run_id = df_pandas["run_id"].iloc[0]

    X = df_pandas[["feature_1", "feature_2", "feature_3"]]
    y = df_pandas["label"]

    # TODO (apartado 4): Separación entre entrenamiento y test con random state 42 y test size 0.3
    ...

    rf = RandomForestRegressor() # Realizar pruebas con distintas tipologías de modelo 
    rf.fit(X_train, y_train)

    predictions = rf.predict(X_test)
    mse = mean_squared_error(y_test, predictions)

    # Registro condicional del modelo si tiene un MSE razonable (mse < 5.0) 
    if mse < 5.0:
        with mlflow.start_run(run_id=run_id) as outer_run:
            # Creamos un run anidado por cada dispositivo con su device_id como nombre.
            experiment_id = outer_run.info.experiment_id
            with mlflow.start_run(run_name=str(device_id), nested=True, experiment_id=experiment_id) as run:
                mlflow.sklearn.log_model(rf, str(device_id))
                mlflow.log_metric("mse", mse)
                # TODO (apartado 5) busca en la documentación de MLflow y registra un tag por cada device usando el device_id, esto será útil para realizar filtrados posteriores. 
                ... 
                
                # Este path nos servirá a futuro para leer el modelo entrenado y aplicarlo
                artifact_uri = f"runs:/{run.info.run_id}/{device_id}"
                return_df = pd.DataFrame([[device_id, n_used, artifact_uri, mse]], 
                                        columns=["device_id", "n_used", "model_path", "mse"])
    else: 
        # TODO (apartado 6): si el error es superior a la cota elegida, crea un dataframe vacío que coincida con el schema de salida sobre la variable return_df
        return_df = ...
    return return_df



## Aplicando la función de pandas a datos agregados:  


In [0]:
# Sección 4: Aplicar la función anterior con applyInPandas agrupando por device_id
# Esto paraleliza el entrenamiento por partición (device_id). Se añade run_id para la ejecución anidada y se cachea el resultado para utilización posterior

with mlflow.start_run(run_name="Entrenamiento por dispositivo") as run:
    run_id = run.info.run_id

    model_directories_df = (df
        .withColumn("run_id", f.lit(run_id))
        .groupby("device_id")
        # TODO (apartado 7): Aplicar la función anterior con applyInPandas agrupando por device_id e indicando el schema previamente definido
        ....
        .cache() # Usaremos cache para reutilizar el DataFrame a posteriori, ¿Antes de ejecutar la celda pregúntate como funciona cache() y si la operación desencadena una acción?
    )

# Combinaremos el resultado al DataFrame original
combined_df = df.join(model_directories_df, on="device_id", how="left")
display(combined_df)

device_id,record_id,feature_1,feature_2,feature_3,label,n_used,model_path,mse
0,0,0.16909051737572756,0.07855672570168681,1.1578523681486637,1.945764807309211,10000,runs:/b997f6fe0fa247ba971c749446e9e37d/0,0.09501251
1,1,0.1401006115074629,1.1786549725659505,1.5909418526078933,3.6907511181864505,10000,runs:/30cfc50d322a47caa51e8952ca14ebf7/1,0.100456566
2,2,0.5487292689854847,0.9540384485910758,2.312818474458484,4.6881434818222125,10000,runs:/f8b6d82877bd44d69f216e829f840871/2,0.09977385
3,3,0.6921770687111917,1.5880553657601024,1.152635258283616,3.4676752441804677,10000,runs:/9961940d756d4678a0155e7d947fe257/3,0.097107545
4,4,0.43473727150673125,1.4380735299576735,1.3214582490082556,3.5522286940537624,10000,runs:/69eb07e5c24d41a5ab2cafbdbb714bb6/4,0.09613026
5,5,0.8466550568398138,0.7282194374755349,2.220938964759944,4.376163238532914,10000,runs:/3b121b6ff44f4d94940b180b8c94212a/5,0.09747307
6,6,0.26812974614268037,0.7638437215542682,0.9058429998387019,2.866543598314112,10000,runs:/c40168ce3dff456fa0c2a34aead641fe/6,0.102718525
7,7,0.23477702719303906,1.248987432268733,0.4921912874192682,2.1745654024421803,10000,runs:/984e50749b994ab19250281db09ea205/7,0.09755363
8,8,0.6121435800327882,1.9723950996713389,1.719198646463564,4.545091593275324,10000,runs:/778b585570ee4711adc99864ba4d0db5/8,0.096298955
9,9,0.4741087239309755,0.9333498118368564,0.08998013369345959,2.3160335887303094,10000,runs:/b3c94a7a1d1d4b95a4eb6affe40de2cc/9,0.09742032



## Modelos ya entrenados:   

En este momento, podéis dirigiros a la pestaña Experimentes en la parte inferior izquierda de la interfaz de Usuario y comprobar que SÍ, los modelos han sido entrenados en un único experimento. Ahora os preguntaréis, ¿cómo podemos utilizarlos?

Vamos a definir una función de pandas y un esquema de retorno para realizar las predicciones con cada modelo también en modo distribuido.

In [0]:
# Sección 5: Aplicación de los modelos entrenados sobre las mismas observaciones originales
# Validación del schema al devolver los tipos de retorno
apply_return_schema = "record_id integer, prediction float"


def apply_model(df_pandas: pd.DataFrame) -> pd.DataFrame:
    if "model_path" not in df_pandas.columns:
        raise ValueError("Falta la columna 'model_path'")

    model_path = df_pandas["model_path"].iloc[0]
    X = df_pandas[["feature_1", "feature_2", "feature_3"]]

    try:
        # TODO (apartado 8): utiliza la columna correspondiente para leer el modelo empleando mlflow y flavor sklearn
        model = 
        ...
    except Exception as e:
        print(f"[ERROR] No se pudo cargar el modelo desde {model_path}: {e}")
        return pd.DataFrame(columns=["record_id", "prediction"])

    prediction = model.predict(X)

    return_df = pd.DataFrame({
        "record_id": df_pandas["record_id"],
        "prediction": prediction
    })
    return return_df

prediction_df = combined_df.groupby("device_id").applyInPandas(apply_model, schema=apply_return_schema)
display(prediction_df)


record_id,prediction
0,1.7665995
10,4.769612
20,4.9085474
30,4.000501
40,3.7202349
50,3.8984017
60,4.322019
70,3.2950315
80,4.002891
90,4.2953854



### Servir múltiples modelos desde un modelo registrado

MLflow permite desplegar modelos como APIs REST en tiempo real. Actualmente, un único modelo en MLflow se sirve desde una instancia (normalmente un contendor, ver VÍDEO SOBRE DESPLIEGUE DE MODELOS CON MLflow). Sin embargo, en algunos casos es necesario servir múltiples modelos desde un único punto de entrada. Imagina 1000 modelos similares que deben usarse con distintas entradas. Ejecutar 1000 endpoints por separado podría desperdiciar recursos, como es el caso, especialmente si algunos de esos modelos se usan con poca frecuencia.

Una solución a este problema es empaquetar varios modelos dentro de un único modelo personalizado, el cual internamente enruta las peticiones al modelo ML adecuado según la entrada recibida, y despliega ese "paquete" de modelos como si fuera un único modelo.

A continuación, vamos a crear un modelo que enrute y agrupe todos los modelos entrenados para cada dispositivo. Por cada fila de datos que se le pase, el modelo determinará el `device_id` y utilizará el modelo correspondiente entrenado para ese dispositivo para hacer la predicción.

Primero, debemos acceder a los modelos correspondientes a cada `device_id`.


In [0]:
# Sección 6: Consulta de modelos entrenados registrados en MLflow a partir del experimento
# Se utiliza `mlflow-experiment` como fuente de datos, filtrando por tag 'device'
experiment_id = run.info.experiment_id

model_df = (spark.read.format("mlflow-experiment")
            .load(experiment_id)
            .filter("tags.device IS NOT NULL")
            .orderBy("end_time", ascending=False)
            .select("tags.device", "run_id")
            .limit(10))

display(model_df)

device,run_id
4,69eb07e5c24d41a5ab2cafbdbb714bb6
2,f8b6d82877bd44d69f216e829f840871
8,778b585570ee4711adc99864ba4d0db5
3,9961940d756d4678a0155e7d947fe257
1,30cfc50d322a47caa51e8952ca14ebf7
5,3b121b6ff44f4d94940b180b8c94212a
6,c40168ce3dff456fa0c2a34aead641fe
9,b3c94a7a1d1d4b95a4eb6affe40de2cc
7,984e50749b994ab19250281db09ea205
0,b997f6fe0fa247ba971c749446e9e37d


Creamos un diccionario mapeando los modelos

In [0]:
# Sección 7: Carga de modelos por device_id en un diccionario, con manejo de errores por si algún modelo falló

device_to_model = {}

# WARNING: collect()
for row in model_df.collect(): 
    # TODO (apartado 9): explica brevemente por qué collect puede ser problemático
    try:
        # Accedemos a la row correspondiente y carga el modelo desde MLflow
        model = mlflow.sklearn.load_model(f"runs:/{row['run_id']}/{row['device']}")
        # TODO (apartado 10): guarda en el mapa asociando cada modelo a su device id
        ...
    except Exception as e:
        print(f"[ERROR] No se pudo cargar el modelo para device={row['device']}: {e}")

Creamos un modelo personalizado que recibe como atributo un mapeo entre identificadores de dispositivo y modelos, y delega la entrada al modelo correspondiente según el device_id. No es necesario modificar el código de la siguiente celda. Dedica algo de tiempo a entener cada uno de los pasos. 

In [0]:
# Sección 8: Creación de un modelo (PythonModel) que enruta la predicción al modelo correcto según device_id
# Este modelo encapsula todos los modelos individuales y los enruta según device_id

from mlflow.pyfunc import PythonModel

class OriginDelegatingModel(PythonModel):
    """
    OriginDelegatingModel es una clase que hereda de mlflow.pyfunc.PythonModel, lo que permite crear un modelo personalizado en MLflow. Este tipo de clase es necesaria cuando quieres definir tu propia lógica de predicción y empaquetarla para que pueda usarse con mlflow.pyfunc.load_model(...).

    Para que funcione correctamente necesitarás implementar el método predict(...) que recibe un DataFrame de pandas y devuelve un array de predicciones.
    """

    def __init__(self, device_to_model_map):
        self.device_to_model_map = device_to_model_map

    def _predict_for_device(self, row: pd.Series) -> float:
        """
        Realiza una predicción para una única fila de entrada, delegando el modelo
        a utilizar en función del `device_id`.

        Este método es útil en escenarios donde se han entrenado múltiples modelos
        personalizados (por ejemplo, uno por dispositivo IoT), y se necesita enrutar 
        la inferencia al modelo correspondiente.

        Args:
            row (pd.Series): Una fila del DataFrame de entrada. Se espera que contenga 
                            al menos una columna `device_id` y las columnas de entrada
                            del modelo (`feature_1`, `feature_2`, `feature_3`).

        Returns:
            float: La predicción del modelo correspondiente al `device_id`. Si no hay
                un modelo disponible para el dispositivo, se devuelve NaN.
        """
        model = self.device_to_model_map.get(str(row["device_id"]))
        if model is None:
            return float("nan")
        data = row[["feature_1", "feature_2", "feature_3"]].to_frame().T
        return model.predict(data)[0]
    
    def predict(self, model_input):
        return model_input.apply(self._predict_for_device, axis=1)


/databricks/python/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:186: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


In [0]:
# Sección 9: Ejemplo de uso del modelo combinado sobre un subconjunto de datos

# TODO (apartado 11): inicializa un objeto de las clase OriginDelegatingModel y aplícalo sobre combined_df, recuerda que el input debe ser un pandas.DataFrame, aplícalo solo para los 20 primeros valores

example_model = ...
predictions = example_model.predict(combined_df.toPandas().head(20))
display(predictions)

A partir de aquí, podemos registrar el modelo para que sea utilizado en la inferencia de todos los device_id desde una única instancia.

In [0]:
from mlflow.models.signature import infer_signature

input_example = combined_df.select("device_id", "feature_1", "feature_2", "feature_3").toPandas().head(5)

# Ejecuta el modelo sobre el ejemplo para obtener la salida
output_example = example_model.predict(input_example)

# Infieres la firma a partir del input y output. Esto es una buena práctica para validar datos entrada/salida al realizar predicciones. 
signature = infer_signature(input_example, output_example)

# Sección 10: Registro del modelo combinado en MLflow para su reutilización o despliegue
with mlflow.start_run():
    model = OriginDelegatingModel(device_to_model)
    # TODO (apartado 12): registra el modelo usando el flavor necesario para modelos customizados. 
    ...(
        "model", 
        python_model=model,
        input_example=input_example,
        signature=signature
        )

/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2025/06/17 06:33:14 WARNING mlflow.models.signature: Failed to infer schema for outputs. Setting schema to `Schema([ColSpec(type=AnyType())]` as default. To see the full traceback, set logging level to DEBUG.
2025/06/17 06:33:14 INFO mlflow

Uploading artifacts:   0%|          | 0/11 [00:00<?, ?it/s]

Uploading /local_disk0/repl_tmp_data/ReplId-1977c-75339-9/tmpdkjjjo2q/model/python_model.pkl:   0%|          |…

In [0]:
# TODO (apartado 13): utiliza la Interfaz de Usuario de Experiments para crear un Modelo llamado iot_model basado en el experimento anterior. Haz DOS capturas de pantalla: 
# 1. Experimento realizado con nested_run en mlflow desplegando los modelos por device_id. Nombre de la captura experiments.png
# 2. Modelo creado desde la UI en la pestaña Models. Nombre de la captura model.png

#  Añade ambas capturas al zip. 

Capturas de ejemplo


![Experimentos](https://github.com/pdehidalgo/dbx-ucm-productivizacion/raw/main/experimentos.png)
![Registro](https://github.com/pdehidalgo/dbx-ucm-productivizacion/raw/main/registro_modelo.png)


In [0]:
# (Último ejericio) TODO (apartado 14, opcional pero evaluable): una vez registrado el modelo, levanta un Endpoint desde la interfaz de usuario llamado iot_endpoint, no olvides escalarlo a zero y borrarlo una vez terminemos la práctica. Utiliza un cliente Python (un Notebook puede valer), Postman o CUrl para realizar una llamada al endpoint levantado. Revisa el material de la documentación interactiva para entender cómo enviar los datos y no recibir un error 500. Añade una captura de pantalla llamada endpoint.png con el resultado final de la llamada al endpoint mostrando la predicción. Añade la captura al zip.

# IMPORTANTE

Un serving endpoint aunque escalado a 0 puede incurrir en costes (del orden de 3-5 euros diarios) por el mantenimiento de la IP pública. Por favor, borra el serving una vez terminada la práctica. 

![Endpoint IOT](https://github.com/pdehidalgo/dbx-ucm-productivizacion/raw/main/endpoint_iot.png)


## En la tarea, hemos trabajado: 

- Cómo entrenar modelos en paralelo por particiones usando Spark y Pandas UDFs.
- Cómo registrar modelos condicionalmente con MLflow.
- Cómo combinar múltiples modelos en un único punto de inferencia.

**IMPORTANTE**: Este patrón es escalable y aplicable a muchos escenarios del mundo real.
